# Lab 2 : OD Matrices analytics

We want to find out which are more likely class of users of car sharing system. For this we compare the OD matrices obtained from different sources. 

## Setup of datasets and visualization

In [11]:
import pymongo as pm
from pprint import pprint
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from datetime import datetime, timezone

### Car sharing - MongoDB

In [3]:
client = pm.MongoClient(
    'bigdatadb.polito.it',
    ssl=True,
    authSource='carsharing',
    username='ictts',
    password='Ict4SM22!',
    tlsAllowInvalidCertificates=True
)

In [4]:
db = client['carsharing']
pb_coll = db['ictts_PermanentBookings']
pb_coll_enjoy= db['ictts_enjoy_PermanentBookings']

In [5]:
list(pb_coll.find().limit(1))

[{'_id': ObjectId('5a3bb3af20c3dd47cf59003d'),
  'init_fuel': 53,
  'vendor': 'car2go',
  'exterior': 'GOOD',
  'vin': 'WME4533421K152664',
  'final_address': 'Corso Regina Margherita, 107, 10122 Torino TO',
  'engineType': 'CE',
  'final_date': datetime.datetime(2017, 11, 1, 0, 19, 45),
  'public_transport': {'duration': -1,
   'arrival_time': -1,
   'distance': -1,
   'arrival_date': -1},
  'final_loc': {'coordinates': [7.68634, 45.07567], 'type': 'Point'},
  'smartPhoneRequired': True,
  'init_time': 1509490770,
  'interior': 'GOOD',
  'init_address': 'Lungo Dora Napoli, 18A, 10152 Torino TO',
  'init_loc': {'coordinates': [7.68486, 45.08276], 'type': 'Point'},
  'driving': {'duration': 234, 'distance': 966},
  'final_time': 1509491985,
  'plate': '288/FF214SJ',
  'city': 'Torino',
  'final_fuel': 53,
  'init_date': datetime.datetime(2017, 10, 31, 23, 59, 30),
  'walking': {'duration': 746, 'distance': 982}}]

In [21]:
print("START:", START)
print("END:", END)

print("Premier document:")
print(pb_coll.find_one({}, {"init_time": 1, "init_date": 1, "driving": 1}))

print("Documents dans la période:",
      pb_coll.count_documents({
          "init_time": {"$gte": START, "$lt": END}
      }))

START: 1509494400
END: 1517443200
Premier document:
{'_id': ObjectId('5a3bb3af20c3dd47cf59003d'), 'init_time': 1509490770, 'driving': {'duration': 234, 'distance': 966}, 'init_date': datetime.datetime(2017, 10, 31, 23, 59, 30)}
Documents dans la période: 0


In [6]:
list(pb_coll_enjoy.find().limit(1))

[{'_id': ObjectId('5a3bb23920c3dd4716519e40'),
  'final_fuel': 56,
  'init_fuel': 57,
  'plate': 'EZ807DD',
  'car_category_id': 8,
  'driving': {'distance': 3948, 'duration': 545},
  'onClick_disabled': False,
  'init_loc': {'coordinates': [7.7029123, 45.07156], 'type': 'Point'},
  'walking': {'distance': 3776, 'duration': 2849},
  'virtual_rental_id': 361384,
  'car_category_type_id': 1,
  'carModelData': [99, 100, 141],
  'virtual_rental_type_id': 2,
  'final_address': 'Via Arcangelo Corelli, 41, 10154 Torino TO',
  'init_address': 'Corso Carlo Luigi Farini, 20, 10153 Torino TO',
  'final_time': 1509491472,
  'public_transport': {'distance': 4034,
   'duration': 1377,
   'arrival_time': 1509493666,
   'arrival_date': datetime.datetime(2017, 11, 1, 0, 47, 46)},
  'init_date': datetime.datetime(2017, 10, 31, 23, 58, 50),
  'final_date': datetime.datetime(2017, 11, 1, 0, 11, 12),
  'city': 'Torino',
  'vendor': 'enjoy',
  'init_time': 1509490730,
  'final_loc': {'coordinates': [7.70973

In [7]:
DoWMornings=pb_coll_enjoy.aggregate([
    {'$project': {
        'hour':{'$hour':'$init_date'},
        'day':{'$dayOfWeek':'$init_date'},
        'init_loc':1,
        'final_loc':1
    }},
    {'$match':{
        'hour': {'$gte':0,'$lt':12},
        'day':{'$gte':2,'$lt':7}
    }} ,
    {'$count':'tot'}
])

In [8]:
result = next(DoWMornings, None)
print(result)

{'tot': 23148}


### UnipolData

In [9]:
df_Uni_Info = pd.read_csv("data/Unipoldata/Info_TO.csv")
print(df_Uni_Info.head())       
print(df_Uni_Info.info())  

df_Uni_Trips = pd.read_csv("data/Unipoldata/Trips_OD_TO.csv")
print(df_Uni_Trips.head())       
print(df_Uni_Trips.info())  

   id_veicolo commerciale  eta_approx genere
0           1           N          47      M
1           2           N          54      M
2           3           N          34      F
3           4           N          81      M
4           5           N          23      M
<class 'pandas.DataFrame'>
RangeIndex: 37949 entries, 0 to 37948
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   id_veicolo   37949 non-null  int64
 1   commerciale  37949 non-null  str  
 2   eta_approx   37949 non-null  int64
 3   genere       37949 non-null  str  
dtypes: int64(2), str(2)
memory usage: 1.2 MB
None
   id_veicolo  id_viaggio        istante_start  lat_start  lon_start  \
0           1           2  2024-08-27 21:15:03     45.069      7.698   
1           1           4  2024-08-27 21:22:47     45.069      7.698   
2           1           5  2024-08-28 08:16:42     45.083      7.675   
3           1           6  2024-08-28 11:47:37     45.

### IMQ

## Question 1

### import of Lab1 filtering

In [26]:
Q = 0.90

In [27]:
def load_wtime(db, coll, carsharing=True):
    if carsharing:
        pipe = [
            {
                "$match": {
                    "driving.duration": {"$gt": 0},
                    "driving.distance": {"$gt": 0}
                }
            },
            {
                "$project": {
                    "_id": 0,
                    "init_time": 1,
                    "final_time": 1,
                    "duration": "$driving.duration",
                    "day": {
                        "$dateToString": {
                            "date": "$init_date",
                            "format": "%Y-%m-%d"
                        }
                    },
                    "week": {"$isoWeek": "$init_date"},
                    "iso_year": {"$isoWeekYear": "$init_date"},
                    "dow": {"$isoDayOfWeek": "$init_date"},
                    "hour": {"$hour": "$init_date"},
                    "vendor": 1,
                    "init_address": 1,
                    "final_address": 1,
                    "plate": 1,
                    "init_loc": 1,
                    "final_loc": 1,
                    "driving": 1,
                    "walking": 1
                }
            }
        ]

    else:
        raise NotImplementedError(
            "Non-carsharing data processing is not implemented yet."
        )

    collection = db[coll] if isinstance(coll, str) else coll
    df = pd.DataFrame(collection.aggregate(pipe, allowDiskUse=True))

    if df.empty:
        print("Warning: MongoDB query returned no documents.")
        return df

    df["duration_min"] = df["duration"] / 60

    df["driving_distance_km"] = df["driving"].apply(
        lambda x: x["distance"] / 1000
        if isinstance(x, dict) else np.nan
    )

    df["walking_duration_min"] = df["walking"].apply(
        lambda x: x["duration"] / 60
        if isinstance(x, dict) else np.nan
    )

    return df



def best_zone(x, q=Q, log=True, threshold=False, min_duration=1.0):
    x = np.sort(np.asarray(x, dtype=float))
    if threshold:
        x = x[x >= min_duration]
    else:
        x = x[x >= 0]
    n = len(x)
    if n == 0:
        return np.nan, np.nan
    k = int(np.ceil(q * n))
    t = np.log(x) if log else x
    widths = t[k - 1:] - t[:n - k + 1]
    i = np.argmin(widths)
    return x[i], x[i + k - 1]


def keep_best_zone(df, q=Q, col="duration_min", log=True, threshold=False):
    a, b = best_zone(df[col].values, q, log, threshold)
    return df[df[col].between(a, b)].copy(), (a, b)


def get_filtered_data(db,coll,q=Q,carsharing=True,verbose=True,return_extra=False):
    raw = load_wtime(db=db,coll=coll,carsharing=carsharing)

    if raw.empty:
        raise ValueError("No data returned by MongoDB.")

    filtered, zone = keep_best_zone(raw, q=q)

    if verbose:
        a, b = zone
        n0, n1 = len(raw), len(filtered)
        percentage = n1 / n0 if n0 else 0

        print(
            f"zone = [{a:.1f}, {b:.1f}] min "
            f"-> {n1}/{n0} gardés ({percentage:.1%})"
        )

    if return_extra:
        return filtered, {"raw": raw, "zone": zone}

    return filtered

In [25]:
df = get_filtered_data(
    db=db,
    coll=pb_coll,
    q=0.90,
    carsharing=True
)

zone = [5.0, 22.0] min -> 35276/39190 gardés (90.0%)
